In [ ]:
import numpy as np
import pymc as pm
import pytensor.tensor as pt

np.random.seed(42)

class Detector:
    def __init__(self, R, r, N):
        self.R = R
        self.r = r
        self.N = N
        self.positions = self._place_pmt()

    def _place_pmt(self):
        points = []
        while len(points) < self.N:
            p = np.random.uniform(-1, 1, size=3)
            length = np.linalg.norm(p)
            if 0 < length <= 1:
                points.append(p / length * self.R)
        return np.array(points)

det = Detector(R=2.0, r=0.1, N=100)

Y_true = 10**5
E_true = 1.0
pos_true = np.array([0.5, -0.3, 0.8])
L_abs = 100.0
a = 0.5

vec_to_pmt = det.positions - pos_true
dist_to_pmt = np.linalg.norm(vec_to_pmt, axis=1)
# НОРМАЛЬ ФЭУ СМОТРИТ ВНУТРЬ: -positions/R. Вектор от ФЭУ к событию: -vec_to_pmt.
# cos = (-pos/R) · (-vec/dist) = (pos/R) · (vec/dist)
cos_theta = np.sum((det.positions / det.R) * (vec_to_pmt / dist_to_pmt[:, np.newaxis]), axis=1)
cos_theta = np.clip(cos_theta, 0, 1)
solid_angle = np.pi * (det.r**2) / (dist_to_pmt**2) * cos_theta
mu_true = E_true * Y_true * (solid_angle / (4 * np.pi)) * np.exp(-dist_to_pmt / L_abs) * a

print(f"mu_true: среднее {mu_true.mean():.2f}, макс {mu_true.max():.2f}, сумма {mu_true.sum():.1f}")
observed_counts = np.random.poisson(mu_true)
print(f"observed: среднее {observed_counts.mean():.2f}, макс {observed_counts.max()}, сумма {observed_counts.sum()}")

with pm.Model() as model:
    E = pm.HalfNormal('E', sigma=2.0)
    x = pm.Normal('x', mu=0, sigma=1.0)
    y = pm.Normal('y', mu=0, sigma=1.0)
    z = pm.Normal('z', mu=0, sigma=1.0)

    pos = pt.stack([x, y, z])
    dist_from_center = pt.sqrt(x**2 + y**2 + z**2)
    pm.Potential('inside_detector', pt.switch(dist_from_center > det.R, -1e3, 0))

    vec_to_pmt = det.positions - pos
    dist_to_pmt = pt.sqrt(pt.sum(vec_to_pmt**2, axis=1))
    safe_dist = pt.maximum(dist_to_pmt, 1e-5)

    cos_theta = pt.sum((det.positions / det.R) * (vec_to_pmt / safe_dist[:, None]), axis=1)
    cos_theta = pt.clip(cos_theta, 0, 1)
    solid_angle = np.pi * (det.r**2) / (safe_dist**2) * cos_theta

    mu = E * Y_true * (solid_angle / (4 * np.pi)) * pt.exp(-safe_dist / L_abs) * a

    counts = pm.Poisson('counts', mu=mu, observed=observed_counts)
    trace = pm.sample(2000, tune=2000, chains=4, target_accept=0.95, progressbar=False)

E_samples = trace.posterior['E'].values.flatten()
x_samples = trace.posterior['x'].values.flatten()
y_samples = trace.posterior['y'].values.flatten()
z_samples = trace.posterior['z'].values.flatten()

print(f"Истинная энергия: {E_true:.3f}")
print(f"Реконструированная энергия: {np.mean(E_samples):.3f} ± {np.std(E_samples):.3f}")
print(f"Истинное положение: {pos_true}")
print(f"Реконструированное положение: [{np.mean(x_samples):.3f}, {np.mean(y_samples):.3f}, {np.mean(z_samples):.3f}]")
print(f"Стд. откл. положения: [{np.std(x_samples):.3f}, {np.std(y_samples):.3f}, {np.std(z_samples):.3f}]")

mu_true: среднее 29.66, макс 97.20, сумма 2966.4
observed: среднее 30.81, макс 108, сумма 3081


Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 2 jobs)
NUTS: [E, x, y, z]
